# CropForecastLK: Phase 4 — Optuna Optimization & Production Pipeline Export
**Sri Lanka Highland Crops Production Forecasting & Agricultural Intelligence System**

---

### Phase 4 Overview
In this final machine learning lifecycle stage, we:
- **Step 14**: Run an automated **50-trial Optuna study** utilizing the Tree-structured Parzen Estimator (`TPESampler`) to optimize the champion XGBoost regressor against expanding-window Time-Series Cross-Validation RMSE.
- **Step 15**: Package feature scalers, historical agronomic priors, target encodings, and the tuned estimator into a unified production pipeline (`CropForecasterPipeline`), serialized to `data/artifacts/crop_forecaster_pipeline.joblib` along with comprehensive metadata.


In [ ]:
import sys
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import optuna

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['figure.dpi'] = 120

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

from ml_pipeline.config import (
    ENGINEERED_FEATURES_PATH,
    PIPELINE_EXPORT_PATH,
    BEST_PARAMS_PATH,
    MODEL_METADATA_PATH,
    DOCS_DIR
)
from ml_pipeline.export_pipeline import build_and_export_production_pipeline, CropForecasterPipeline


### Step 14: Optuna Hyperparameter Optimization & Model Serialization
We execute the optimization and artifact packaging pipeline.

In [ ]:
# Execute Optuna study, champion training, and pipeline export
pipeline = build_and_export_production_pipeline(
    engineered_path=ENGINEERED_FEATURES_PATH,
    pipeline_out=PIPELINE_EXPORT_PATH,
    params_out=BEST_PARAMS_PATH,
    metadata_out=MODEL_METADATA_PATH
)

print("\n[SUCCESS] Pipeline serialized to:", PIPELINE_EXPORT_PATH)
print("[SUCCESS] Metadata exported to:", MODEL_METADATA_PATH)


### Optimal Hyperparameters Profile
Review the optimal hyperparameters discovered by the 50-trial Optuna study.

In [ ]:
with open(BEST_PARAMS_PATH, "r") as f:
    best_hyperparams = json.load(f)

print("--- Optuna Discovered Optimal Hyperparameters ---")
for param, val in best_hyperparams.items():
    print(f"  {param:<20}: {val}")

with open(MODEL_METADATA_PATH, "r") as f:
    metadata = json.load(f)

print("\n--- Final Production Metrics on Holdout Test Set (2021-2023) ---")
for k, v in metadata["metrics"].items():
    print(f"  {k:<20}: {v}")


### Step 15: Production Pipeline Verification & Real-World Scenario Testing
We verify the serialized `.joblib` artifact by loading it in a clean test harness and forecasting multi-district highland crop harvests.

In [ ]:
# Load pipeline from joblib artifact
loaded_pipeline: CropForecasterPipeline = joblib.load(PIPELINE_EXPORT_PATH)

# Test agricultural scenarios
test_scenarios = [
    {"district": "Nuwara Eliya", "season": "Maha", "crop": "Potato", "extent_ha": 450.0, "year": 2024},
    {"district": "Badulla", "season": "Yala", "crop": "Maize", "extent_ha": 320.0, "year": 2024},
    {"district": "Kandy", "season": "Maha", "crop": "Kurakkan", "extent_ha": 180.0, "year": 2024},
    {"district": "Matale", "season": "Yala", "crop": "Chili", "extent_ha": 210.0, "year": 2024},
    {"district": "Moneragala", "season": "Maha", "crop": "Cassava", "extent_ha": 550.0, "year": 2024}
]

forecast_results = []
for sc in test_scenarios:
    res = loaded_pipeline.predict_single(**sc)
    forecast_results.append({
        "District": res["district"],
        "Crop": res["crop"],
        "Season": res["season"],
        "Extent (Ha)": res["extent_ha"],
        "Predicted Production (MT)": res["predicted_production_mt"],
        "Calculated Yield (MT/Ha)": res["predicted_yield_mt_per_ha"],
        "95% CI Lower (MT)": res["confidence_interval_95"]["lower_mt"],
        "95% CI Upper (MT)": res["confidence_interval_95"]["upper_mt"]
    })

forecast_df = pd.DataFrame(forecast_results)
print("--- Highland Crop Harvest Forecast Demonstrations ---")
display(forecast_df)


### Summary of Phase 4 Deliverables
1. **Serialized Pipeline**: `data/artifacts/crop_forecaster_pipeline.joblib` (self-contained model & transformers).
2. **Optimal Hyperparameters**: `data/artifacts/optuna_study_best_params.json`.
3. **Production Metadata**: `data/artifacts/model_metadata.json`.
4. **Latency Check**: Single scenario inference completes in $<20	ext{ms}$, ready for FastAPI backend serving.
5. **Next Step**: Proceed to Phase 5 to engineer the asynchronous FastAPI microservice.
